# From EEG sensors to brain sources, guided by fMRI
### A hands-on, synthetic re-creation of Henson et al. (2019), *Multimodal integration of M/EEG and f/MRI data in SPM12*, using MNE-Python

This notebook builds one **fake subject** whose brain activity we know exactly (the "ground truth"). We then run the same pipeline as the paper and check every step against that truth. Because we know the answer, you can see what each step adds.

| # | Step in this notebook | Paper section | What it gives you |
|---|---|---|---|
| 1 | Build the subject: head, electrodes, cortex | Getting started / Create head model | The geometry everything else depends on |
| 2 | Ground truth: what the brain really does | (not in paper, only possible with simulation) | The answer key |
| 3 | Simulate the raw recording | The raw FIF file | Realistic messy data |
| 4 | Preprocessing: downsample, filter, re-reference | Preprocessing M/EEG data | Clean continuous data |
| 5 | Epoching, artifact rejection, averaging, contrast | Epoch / Artifact detection / Average / Contrast | ERPs and the N170 face effect |
| 6 | Forward model (lead field **L**) | Source reconstruction: forward model | How each brain point appears at each electrode |
| 7 | The inverse problem in miniature | Motivation and background | Why the problem has no single answer |
| 8 | Preparing data for inversion | (inside SPM's inversion) | Whitened, reduced data |
| 9 | The Bayesian (PEB) inversion engine | Model inversion | One engine for all methods |
| 10 | Inversion 1: Minimum norm (MNM / IID) | Model inversion | A smooth, blurry answer |
| 11 | Inversion 2: Multiple Sparse Priors (MSP) | Model inversion | A focal answer |
| 12 | Simulated fMRI and fMRI priors | Group and fMRI optimized source reconstruction | "Where" hints from fMRI |
| 13 | Inversions 3 & 4: MNM + fMRI, MSP + fMRI | Group and fMRI optimized source reconstruction | Sharper maps; false hints switched off |
| 14 | Model evidence: which assumption fits best? | Model comparison | A principled way to choose |
| 15 | Source power in a time-frequency window + statistics | Time-frequency contrasts / Group statistics | Figure 8 / 9 style maps |

**Simplifications.** These keep everything running in a few minutes on a laptop with no downloads:
* EEG only (64 channels, BioSemi layout). The paper also fuses MEG; the logic is identical, you just stack more rows into **L**.
* The head is a 4-layer **sphere** (brain, CSF, skull, scalp) instead of a head shape from MRI.
* The "cortex" is ~1,200 points on a shell inside the sphere instead of a folded surface with ~8,000 points.
* One subject, so statistics compare **trials** (faces vs scrambled) instead of comparing **subjects**.

Requirements: `pip install mne>=1.6 numpy scipy matplotlib`

In [ ]:
# !pip install "mne>=1.6"      # uncomment if MNE is not installed

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
import mne

mne.set_log_level('ERROR')
rng = np.random.default_rng(15)                     # "subject 15", like the paper
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 9})
print('MNE version:', mne.__version__)

### Helper functions
Small tools used throughout: make points on a sphere, define a "patch" of cortex, and draw the cortex as a flat map seen **from behind the head**. The face areas are at the back of the brain, so this view puts them in the middle of the picture. Unlike SPM's bottom view in Figure 8, left is left and right is right here.

In [ ]:
def fibonacci_sphere(n, radius):
    """n roughly evenly spaced points on a sphere."""
    i = np.arange(n) + 0.5
    phi = np.arccos(1 - 2 * i / n)
    theta = np.pi * (1 + 5 ** 0.5) * i
    return radius * np.c_[np.cos(theta) * np.sin(phi), np.sin(theta) * np.sin(phi), np.cos(phi)]

def nearest(rr, direction):
    """Index of the cortex point closest to a given direction from the head centre."""
    d = np.asarray(direction, float); d /= np.linalg.norm(d)
    return int(np.argmax(rr @ d / np.linalg.norm(rr, axis=1)))

def patch(rr, centre, sigma=0.006, cutoff=2.5):
    """A smooth bump of cortex around point `centre` (weights 0..1)."""
    d = np.linalg.norm(rr - rr[centre], axis=1)
    w = np.exp(-d ** 2 / (2 * sigma ** 2)); w[d > cutoff * sigma] = 0
    return w

def flat(r, centre=(0, -0.75, 0.66)):
    """Azimuthal projection of 3D points, viewed from `centre` direction (default: behind & above)."""
    e3 = np.asarray(centre, float); e3 /= np.linalg.norm(e3)
    e1 = np.array([1., 0, 0]); e1 -= (e1 @ e3) * e3; e1 /= np.linalg.norm(e1)   # subject's right
    e2 = np.cross(e3, e1)                                                       # up
    u = r / np.linalg.norm(r, axis=1, keepdims=True)
    th = np.arccos(np.clip(u @ e3, -1, 1)); ph = np.arctan2(u @ e2, u @ e1)
    return th * np.cos(ph), th * np.sin(ph)

def clusters(rr, mask, radius=0.009):
    """Group neighbouring True points into clusters (lists of indices)."""
    idx = np.where(mask)[0]
    if len(idx) == 0:
        return []
    d = np.linalg.norm(rr[idx, None] - rr[None, idx], axis=2)
    n, lab = connected_components(csr_matrix(d < radius), directed=False)
    return sorted([idx[lab == k] for k in range(n)], key=len, reverse=True)

def plot_cortex(ax, values=None, title='', cmap='hot_r', vmin=None, vmax=None,
                mask=None, show_truth=True, s=16, colorbar=True):
    """Draw a value per cortex point on the flat 'view from behind' map."""
    ax.scatter(PX, PY, c='0.88', s=s, linewidths=0)
    sc = None
    if values is not None:
        v = np.asarray(values, float)
        if mask is not None:
            v = np.where(mask, v, np.nan)
        sc = ax.scatter(PX, PY, c=v, s=s, cmap=cmap, vmin=vmin, vmax=vmax, linewidths=0)
        if colorbar:
            plt.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)
    if show_truth:
        for name in FACE_AREAS:
            i = REGION_IDX[name]
            ax.scatter(PX[i], PY[i], s=170, facecolors='none', edgecolors='tab:blue', linewidths=1.3)
    ax.text(-1.45, 0.0, 'L', fontsize=11, ha='center', va='center')
    ax.text(1.45, 0.0, 'R', fontsize=11, ha='center', va='center')
    ax.set_xlim(-1.6, 1.6); ax.set_ylim(-1.55, 1.55)
    ax.set_aspect('equal'); ax.axis('off'); ax.set_title(title)
    return sc

---
## Step 1. Build the subject: head, electrodes and cortex
**Paper:** "Getting started" and "Create head model" (meshes from the T1 MRI, fiducials, coregistration).

**What happens here.** We create the three geometric ingredients of any source analysis:
1. **Electrodes**: 64 EEG positions (BioSemi layout), placed on the scalp. In the paper these come from the Polhemus digitizer and are **coregistered** to the MRI using the nasion and ear fiducials. Here head and electrodes already share one coordinate system, so coregistration is perfect by construction.
2. **Head layers**: brain, CSF, skull, scalp. The skull matters a lot for EEG because it blurs the electric field. The paper builds these surfaces (meshes) from the subject's MRI; we use spheres.
3. **Cortex (source space)**: the places where brain sources are allowed to be. Each point is one possible "current dipole" with a fixed orientation (perpendicular to the cortex). The paper uses ~8,000 points from the MRI; we use ~1,200 on a shell.

**Why it matters.** Every later step depends on knowing *where the sensors are relative to the brain*. If coregistration is off by 1 cm, every source map is off too.

**Information added:** the geometry of this subject.

In [ ]:
# --- 1a. Electrodes: BioSemi-64 layout, re-centred and placed on a 9 cm scalp sphere
HEAD_R = 0.09                                   # scalp radius in metres
montage = mne.channels.make_standard_montage('biosemi64')
ch_pos = montage.get_positions()['ch_pos']
ch_names = list(ch_pos)
P = np.array([ch_pos[n] for n in ch_names])
A = np.c_[2 * P, np.ones(len(P))]                # least-squares sphere fit to find the head centre
centre = np.linalg.lstsq(A, (P ** 2).sum(1), rcond=None)[0][:3]
U = (P - centre) / np.linalg.norm(P - centre, axis=1, keepdims=True)
el_pos = HEAD_R * U
dig = mne.channels.make_dig_montage(ch_pos=dict(zip(ch_names, el_pos)), coord_frame='head')

SF_RAW = 1000.                                  # recording sampling rate (the paper: 1100 Hz)
info = mne.create_info(ch_names + ['VEOG'], sfreq=SF_RAW, ch_types=['eeg'] * 64 + ['eog'])
info.set_montage(dig)
info_eeg = mne.pick_info(info, mne.pick_types(info, eeg=True))
SPHERE = (0., 0., 0., HEAD_R)                    # used for topomap drawing

# --- 1b. Cortex: points on a 7 cm shell (upper part of the head), normals point outwards
cortex_all = fibonacci_sphere(2000, 0.07)
cortex_rr = cortex_all[cortex_all[:, 2] > -0.015]
cortex_nn = cortex_rr / np.linalg.norm(cortex_rr, axis=1, keepdims=True)
src = mne.setup_volume_source_space(pos=dict(rr=cortex_rr, nn=cortex_nn),
                                    sphere=SPHERE, sphere_units='m', mindist=0.)
rr = src[0]['rr'][src[0]['vertno']]              # the source points MNE kept
nn = rr / np.linalg.norm(rr, axis=1, keepdims=True)
N = len(rr)
PX, PY = flat(rr)                                # 2D coordinates for plotting

# --- 1c. Named brain regions (directions from the head centre: x=right, y=front, z=up)
REGION_DIRS = {'V1':   (0.0, -1.0, 0.35),       # early visual cortex
               'rOFA': (0.5, -0.85, 0.30),      # right occipital face area
               'lOFA': (-0.5, -0.85, 0.30),     # left occipital face area
               'rFFA': (0.8, -0.50, 0.20)}      # right fusiform face area
REGION_IDX = {k: nearest(rr, v) for k, v in REGION_DIRS.items()}
REGION_W = {k: patch(rr, i) for k, i in REGION_IDX.items()}
FACE_AREAS = ['rOFA', 'lOFA', 'rFFA']
print(f'{len(ch_names)} EEG electrodes, {N} cortex source points')
for k, i in REGION_IDX.items():
    print(f'  {k:5s} centre at {np.round(rr[i]*1000).astype(int)} mm, {int((REGION_W[k]>0).sum())} points')

In [ ]:
fig = plt.figure(figsize=(13, 4.3))
# 3D view
ax = fig.add_subplot(1, 3, 1, projection='3d')
u, v = np.mgrid[0:2*np.pi:30j, 0:np.pi/1.7:15j]
for r_, col in [(HEAD_R, '0.6'), (HEAD_R * 0.92, 'tab:orange')]:
    ax.plot_wireframe(r_*np.cos(u)*np.sin(v), r_*np.sin(u)*np.sin(v), r_*np.cos(v), color=col, lw=0.3, alpha=0.5)
ax.scatter(*rr.T, s=2, c='0.5')
for k in REGION_IDX:
    m = REGION_W[k] > 0
    ax.scatter(*rr[m].T, s=8, c='tab:green' if k == 'V1' else 'tab:red')
ax.scatter(*el_pos.T, s=18, c='k')
ax.view_init(18, -110); ax.set_axis_off(); ax.set_title('Head (grey = scalp, orange = skull),\ncortex points, electrodes (black)')

# electrode layout from above
ax = fig.add_subplot(1, 3, 2)
ex, ey = flat(el_pos, centre=(0, 0, 1))
ax.scatter(ex, ey, c='k', s=10)
for n, x_, y_ in zip(ch_names, ex, ey):
    ax.text(x_, y_ + 0.04, n, fontsize=6, ha='center')
ax.add_patch(plt.Circle((0, 0), np.pi/2, fill=False, color='0.6'))
ax.plot([-.12, 0, .12], [np.pi/2, np.pi/2 + .15, np.pi/2], color='0.6')   # nose
ax.set_aspect('equal'); ax.axis('off'); ax.set_title('64 electrodes seen from above (nose up)')

# cortex map from behind with regions
ax = fig.add_subplot(1, 3, 3)
lab = np.zeros(N)
for j, k in enumerate(REGION_IDX):
    lab[REGION_W[k] > 0] = j + 1
plot_cortex(ax, np.where(lab > 0, lab, np.nan), 'Cortex seen from behind, with the 4 regions',
            cmap='tab10', vmin=0.5, vmax=10.5, colorbar=False, show_truth=False)
for k, i in REGION_IDX.items():
    ax.text(PX[i], PY[i] - 0.17, k, ha='center', fontsize=8)
plt.tight_layout(); plt.show()

---
## Step 2. Ground truth: what our fake brain really does
**Paper:** not in the paper. With real data, nobody knows the truth. That is exactly why simulation helps you learn.

**What happens here.** We decide the "answer key":
* **V1** (early visual cortex) responds at **~100 ms** to *every* picture, faces or scrambled.
* **rOFA and lOFA** respond at **~150 ms**, strongly to faces and weakly to scrambled faces.
* **rFFA** responds at **~175 ms**, strongly to faces and very weakly to scrambled faces.

The responses are short bursts at 8–15 Hz, so they fall inside the paper's analysis window (**100–250 ms, 10–20 Hz**). Each trial also gets random amplitude and latency jitter, because real brains are never identical twice.

**Why it matters.** At the end, a good method should find **rOFA, lOFA and rFFA** for *faces > scrambled*, and **not** V1 (V1 responds equally, so the contrast cancels it).

In [ ]:
def gabor(t, t0, f, s):
    return np.exp(-(t - t0) ** 2 / (2 * s ** 2)) * np.cos(2 * np.pi * f * (t - t0))

# (latency s, frequency Hz, width s) and amplitude for faces / scrambled
RESP = {'V1':   dict(t0=0.100, f=8,  s=0.025, face=1.0, scr=1.0),
        'rOFA': dict(t0=0.150, f=13, s=0.030, face=1.0, scr=0.3),
        'lOFA': dict(t0=0.150, f=13, s=0.030, face=1.0, scr=0.3),
        'rFFA': dict(t0=0.175, f=15, s=0.030, face=1.2, scr=0.2)}

tt = np.arange(-0.1, 0.5, 0.001)
fig, axes = plt.subplots(1, 4, figsize=(13, 2.6), sharey=True)
for ax, (k, p) in zip(axes, RESP.items()):
    ax.plot(tt * 1000, p['face'] * gabor(tt, p['t0'], p['f'], p['s']), label='faces', color='tab:red')
    ax.plot(tt * 1000, p['scr'] * gabor(tt, p['t0'], p['f'], p['s']), label='scrambled', color='tab:olive')
    ax.axvspan(100, 250, color='0.92', zorder=0); ax.set_title(k); ax.set_xlabel('time (ms)')
axes[0].set_ylabel('source strength (relative)'); axes[0].legend(fontsize=7)
fig.suptitle('Ground truth source activity (grey band = analysis window 100–250 ms)', y=1.04)
plt.show()

---
## Step 3. Simulate the raw recording
**Paper:** the raw `*_meg.fif` file: one continuous recording per run, with trigger codes.

**What happens here.** We create ~4 minutes of continuous 64-channel EEG at 1000 Hz containing:
* **The brain signals** from Step 2, sent to the electrodes through the forward model (Step 6 explains this).
* **Background brain activity**: 40 random cortex points producing 1/f noise. This is "brain noise", the main thing that hides small responses.
* **Sensor noise**, **slow drifts** (sweat and electrode movement), **50 Hz mains noise**, and **eye blinks** (big on the VEOG channel and frontal electrodes).
* **Events**: 150 trials (50 famous, 50 unfamiliar, 50 scrambled) in random order, with the paper's trigger codes 5 / 13 / 17.

We need the forward model to simulate, so we compute it here and explain it properly in Step 6.

In [ ]:
# Forward model (explained in Step 6)
sphere_model = mne.make_sphere_model(r0=(0., 0., 0.), head_radius=HEAD_R)
identity = mne.transforms.Transform('head', 'mri')            # head and MRI share one coordinate system here
fwd = mne.make_forward_solution(info, trans=identity, src=src, bem=sphere_model, meg=False, eeg=True)
assert list(fwd['info']['ch_names']) == ch_names
assert np.allclose(fwd['source_rr'], rr)
L_free = fwd['sol']['data']                                  # channels x (3 * sources), V per A*m
L_raw = np.einsum('cik,ik->ci', L_free.reshape(len(ch_names), N, 3), nn) * 1e-3   # µV per nAm, fixed orientation
print('Lead field:', L_raw.shape, '(electrodes x cortex points)')

In [ ]:
CONDS = {'Famous': 5, 'Unfamiliar': 13, 'Scrambled': 17}
N_PER = 50
codes = rng.permutation(np.repeat(list(CONDS.values()), N_PER))
soa = rng.uniform(1.4, 1.8, len(codes))
onsets = 2.0 + np.concatenate([[0], np.cumsum(soa[:-1])])          # seconds
n_times = int((onsets[-1] + 2.0) * SF_RAW)
events = np.c_[(onsets * SF_RAW).astype(int), np.zeros(len(codes), int), codes]
times = np.arange(n_times) / SF_RAW
n_ch = len(ch_names)

# --- brain signal: one time course per region, sent through the lead field
patterns = np.stack([L_raw @ REGION_W[k] for k in RESP], axis=1)        # channels x 4
AMP_nAm = 6.0 / np.abs(L_raw @ REGION_W['rFFA']).max()                   # scale: ~5 µV peak, like an N170
src_tc = np.zeros((len(RESP), n_times))
t_ev = np.arange(-0.1, 0.6, 1 / SF_RAW)
for onset, code_ in zip(events[:, 0], codes):
    is_face = code_ in (5, 13)
    for j, (k, p) in enumerate(RESP.items()):
        amp = (p['face'] if is_face else p['scr']) * (1 + 0.2 * rng.standard_normal())
        jit = 0.005 * rng.standard_normal()
        seg = slice(onset - 100, onset - 100 + len(t_ev))
        src_tc[j, seg] += AMP_nAm * amp * gabor(t_ev, p['t0'] + jit, p['f'], p['s'])
brain = patterns @ src_tc
print(f'Each face-area point peaks at ~{AMP_nAm:.1f} nAm (x patch weight)')

# --- background brain noise: 40 random cortex points with 1/f spectra
far = np.min([np.linalg.norm(rr - rr[i], axis=1) for i in REGION_IDX.values()], axis=0) > 0.02
bg_idx = rng.choice(np.where(far)[0], 40, replace=False)
f = np.fft.rfftfreq(n_times, 1 / SF_RAW)
bg_src = np.fft.irfft(np.fft.rfft(rng.standard_normal((40, n_times)), axis=1) / np.maximum(f, 1.0) ** 0.8,
                      n=n_times, axis=1)
bg = L_raw[:, bg_idx] @ bg_src
bg *= 4.0 / bg.std()

# --- non-brain artefacts
drift = np.cumsum(rng.standard_normal((n_ch, n_times)), axis=1) * 0.04
drift -= drift.mean(1, keepdims=True)
line = rng.uniform(2, 6, (n_ch, 1)) * np.sin(2 * np.pi * 50 * times)[None, :]
sensor = 1.5 * rng.standard_normal((n_ch, n_times))

# --- eye blinks: in ~20% of trials (inside the epoch) and some between trials
blink_t = list(onsets[rng.random(len(onsets)) < 0.2] + rng.uniform(0.0, 0.4))
blink_t += list(rng.uniform(1, times[-1] - 1, 25))
blink = np.zeros(n_times)
for bt in blink_t:
    blink += 300 * np.exp(-(times - bt) ** 2 / (2 * 0.05 ** 2))
fpz = el_pos[ch_names.index('Fpz')]
blink_gain = 0.8 * np.exp(-np.linalg.norm(el_pos - fpz, axis=1) / 0.03)
eeg = brain + bg + drift + line + sensor + np.outer(blink_gain, blink)
veog = blink + 3 * rng.standard_normal(n_times) + drift[0] * 0.3

raw = mne.io.RawArray(np.vstack([eeg, veog]) * 1e-6, info)            # MNE stores volts
print(raw)

In [ ]:
def plot_traces(ax, data_uv, t, names, title, spacing=60, events_t=None):
    for j, (d, n) in enumerate(zip(data_uv, names)):
        ax.plot(t, d - d.mean() - j * spacing, lw=0.6)
        ax.text(t[0] - 0.02 * (t[-1] - t[0]), -j * spacing, n, ha='right', va='center', fontsize=7)
    if events_t is not None:
        for et, c in events_t:
            ax.axvline(et, color='0.7', lw=0.7, ls='--')
            ax.text(et, spacing * 0.9, {5: 'Fam', 13: 'Unf', 17: 'Scr'}[c], fontsize=6, ha='center')
    ax.set_yticks([]); ax.set_xlabel('time (s)'); ax.set_title(title)

show = ['Fp1', 'Fpz', 'Fp2', 'Fz', 'Cz', 'Pz', 'P8', 'PO8', 'O1', 'Oz', 'O2']
pick = [ch_names.index(n) for n in show]
t0 = blink_t[0] - 4; win = (raw.times > t0) & (raw.times < t0 + 8)
ev_in = [(e / SF_RAW, c) for e, c in zip(events[:, 0], codes) if t0 < e / SF_RAW < t0 + 8]
d_all = raw.get_data() * 1e6

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw=dict(width_ratios=[2, 1]))
plot_traces(axes[0], np.vstack([d_all[pick][:, win], d_all[-1:, win] / 4]), raw.times[win],
            show + ['VEOG/4'], 'Raw continuous EEG (cf. paper Figure 3)', spacing=60, events_t=ev_in)
fpsd, psd = signal.welch(d_all[:n_ch], fs=SF_RAW, nperseg=4096)
axes[1].semilogy(fpsd, psd.mean(0)); axes[1].set_xlim(0, 120)
axes[1].set_xlabel('frequency (Hz)'); axes[1].set_ylabel('power (µV²/Hz)')
axes[1].set_title('Spectrum: drift at 0 Hz, mains at 50 Hz')
plt.tight_layout(); plt.show()

**How to read it.** The traces wander slowly (drift), the frontal channels and VEOG show big blink bumps, and the spectrum has a sharp spike at 50 Hz. You **cannot** see the face response by eye: it is buried under background brain activity, drift and blinks. The next steps dig it out.

---
## Step 4. Preprocessing: downsample, filter, re-reference
**Paper:** "Downsample", "Filter", "Montage" (average reference).

| Operation | Paper setting | Why |
|---|---|---|
| Downsample 1000 → 200 Hz | 1100 → 200 Hz | We don't care about >100 Hz; 5x less data makes everything faster. Includes anti-alias filtering. |
| High-pass 1 Hz | Butterworth, order 5, zero-phase | Removes slow drift. Done on **continuous** data so filter ringing doesn't appear in every trial. |
| Low-pass 40 Hz | Butterworth, order 5, zero-phase | Removes 50 Hz mains and muscle noise; ERPs live below 40 Hz. |
| Average reference | "Create average reference montage" | Each channel minus the mean of all channels. Removes the influence of the recording reference. |

"Zero-phase" means the filter runs forwards and then backwards, so peaks don't shift in time. That matters because we care about *when* things happen.

**Information added:** none. Preprocessing only **removes** things we don't want, so the brain signal becomes visible.

In [ ]:
SF = 200.
raw_pre = raw.copy().resample(SF)
events_200 = events.copy()
events_200[:, 0] = np.round(events[:, 0] * SF / SF_RAW).astype(int)

iir = dict(order=5, ftype='butter')
raw_pre.filter(l_freq=1., h_freq=None, picks=['eeg', 'eog'], method='iir', iir_params=iir)
raw_pre.filter(l_freq=None, h_freq=40., picks=['eeg', 'eog'], method='iir', iir_params=iir)
raw_pre.set_eeg_reference('average', projection=False)

d_pre = raw_pre.get_data() * 1e6
win2 = (raw_pre.times > t0) & (raw_pre.times < t0 + 8)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw=dict(width_ratios=[2, 1]))
plot_traces(axes[0], np.vstack([d_pre[pick][:, win2], d_pre[-1:, win2] / 4]), raw_pre.times[win2],
            show + ['VEOG/4'], 'After downsampling, 1–40 Hz filtering, average reference', spacing=40, events_t=ev_in)
f2, psd2 = signal.welch(d_pre[:n_ch], fs=SF, nperseg=1024)
axes[1].semilogy(fpsd, psd.mean(0), label='raw', alpha=0.6)
axes[1].semilogy(f2, psd2.mean(0), label='preprocessed')
axes[1].axvspan(1, 40, color='0.92', zorder=0); axes[1].set_xlim(0, 120); axes[1].legend()
axes[1].set_xlabel('frequency (Hz)'); axes[1].set_title('Spectrum before / after (grey = kept band)')
plt.tight_layout(); plt.show()

**How to read it.** The drift is gone, the traces are flat around zero, and the 50 Hz spike is removed. The blinks are **still there**: they are slow and big, so filtering cannot remove them. That is the job of the next step.

---
## Step 5. Epoching, artifact rejection, averaging and contrasts
**Paper:** "Epoch", "Artifact detection", "Average", "Contrast conditions".

1. **Epoch:** cut out −100 to +500 ms around each picture onset. One continuous recording becomes 150 short trials. No baseline correction, because the high-pass filter already removed the drift (same choice as the paper).
2. **Artifact rejection:** mark trials where the **VEOG** exceeds a threshold (paper: 200 µV) as bad. These are the blink trials. (MNE checks *peak-to-peak* amplitude; SPM checks absolute value. Same idea.)
3. **Average:** average the good trials of each condition. Random noise has no consistent sign across trials, so it cancels out; the brain response is time-locked to the picture, so it survives. With *n* trials, noise shrinks by about √n.
4. **Contrast:** Faces − Scrambled = 0.5 × Famous + 0.5 × Unfamiliar − 1 × Scrambled (the paper's `[0.5 0.5 −1]`). This removes everything common to all pictures, like V1's response to any image.

**Information added:** the **event-related potential (ERP)**, and the **N170 face effect** with its timing and scalp distribution.

In [ ]:
epochs = mne.Epochs(raw_pre, events_200, event_id=CONDS, tmin=-0.1, tmax=0.5,
                    baseline=None, reject=dict(eog=200e-6), preload=True)
n_bad = len(events) - len(epochs)
print(f'Rejected {n_bad} of {len(events)} trials because of blinks')
for c in CONDS:
    print(f'  {c:10s}: {len(epochs[c])} good trials')

ev = {c: epochs[c].average() for c in CONDS}
faces = mne.combine_evoked([ev['Famous'], ev['Unfamiliar']], weights=[0.5, 0.5])
diff = mne.combine_evoked([faces, ev['Scrambled']], weights=[1, -1])

In [ ]:
t_ms = epochs.times * 1000
fig = plt.figure(figsize=(13, 7))
# (a) butterfly of each condition
for j, c in enumerate(CONDS):
    ax = fig.add_subplot(2, 4, j + 1)
    ax.plot(t_ms, ev[c].data.T * 1e6, color='0.4', lw=0.4)
    ax.set_title(f'{c} (n={ev[c].nave})'); ax.set_xlabel('ms'); ax.axvline(0, color='k', lw=0.5)
    if j == 0: ax.set_ylabel('µV (all 64 channels)')
# (b) single channel, cf. paper Figure 5A
ax = fig.add_subplot(2, 4, 4)
ci = int(np.argmax(np.abs(diff.data[:n_ch, np.argmin(np.abs(diff.times - 0.175))])))   # channel with the biggest face effect
best_ch = ch_names[ci]
for c, col in zip(CONDS, ['tab:blue', 'tab:red', 'tab:olive']):
    ax.plot(t_ms, ev[c].data[ci] * 1e6, label=c, color=col)
ax.axvline(0, color='k', lw=0.5); ax.legend(fontsize=7); ax.set_title(f'Channel {best_ch} (cf. Fig 5A)'); ax.set_xlabel('ms')
# (c) one single trial vs. the average, to show what averaging buys
ax = fig.add_subplot(2, 4, 5)
ax.plot(t_ms, epochs['Famous'].get_data(picks='eeg')[0, ci] * 1e6, color='0.6', lw=0.7, label='one trial')
ax.plot(t_ms, ev['Famous'].data[ci] * 1e6, color='tab:blue', label='average')
ax.legend(fontsize=7); ax.set_title(f'Why we average ({best_ch})'); ax.set_xlabel('ms')
# (d) topographies of the face - scrambled difference, cf. paper Figure 5B
vmax = np.abs(diff.data).max() * 1e6
for j, tm in enumerate([0.100, 0.150, 0.175]):
    ax = fig.add_subplot(2, 4, 6 + j)
    k = np.argmin(np.abs(diff.times - tm))
    im, _ = mne.viz.plot_topomap(diff.data[:n_ch, k] * 1e6, info_eeg, axes=ax, show=False,
                                 vlim=(-vmax, vmax), cmap='RdBu_r', sphere=SPHERE)
    ax.set_title(f'Faces − Scrambled\n{int(tm*1000)} ms')
plt.colorbar(im, ax=ax, fraction=0.05, label='µV')
plt.tight_layout(); plt.show()

**How to read it.**
* Single trials are noisy; the average is clean. That is the power of averaging.
* At **100 ms** the difference map is almost empty: V1 responds the same to faces and scrambled faces, so the contrast cancels it.
* At **150–175 ms** a strong difference appears over the **back of the head, stronger on the right**: our "N170".

**The big limitation:** this is a picture of the **scalp**, not the brain. The blob tells us *roughly* where to look, but not *which* brain areas made it. Two nearby areas (OFA and FFA) produce overlapping, blurred scalp patterns. To go from scalp to brain we need the forward and inverse models.

---
## Step 6. The forward model: the lead field **L**
**Paper:** "Source reconstruction: motivation and background" and "Create head model" (EEG BEM / MEG single shell).

**The question it answers:** *"If one cortex point were active with strength 1 nAm, what would each electrode read?"*

Physics (Maxwell's equations) plus the head geometry (where the scalp, skull and electrodes are, and how well each tissue conducts) give the answer. Asking this for every cortex point fills a big table, the **lead field matrix**:

$$\underbrace{Y}_{\text{electrodes}\times\text{time}} = \underbrace{L}_{\text{electrodes}\times\text{cortex points}}\;\underbrace{J}_{\text{cortex points}\times\text{time}} + \underbrace{\varepsilon}_{\text{noise}}$$

* Column *i* of **L** is the **scalp map produced by cortex point *i*** alone.
* The model is **linear**: two active points produce the sum of their two maps.
* Here **L** is 64 × ~1,200. In the paper it is ~370 sensors (EEG + two MEG types) × ~8,000 points.

**Why it matters.** The forward model is the dictionary that translates between "brain language" and "sensor language". The inverse step uses it in reverse.

**Information added:** **anatomy and physics**, the only bridge from scalp to brain. This is what the structural MRI contributes in the paper.

Because our data use the average reference, we apply the same average reference to **L** (subtract the mean over electrodes from each column). Data and model must always be referenced the same way.

In [ ]:
L = L_raw - L_raw.mean(axis=0, keepdims=True)          # average reference, like the data

fig = plt.figure(figsize=(13, 6.5))
ax = fig.add_subplot(2, 4, (1, 5))
order = np.argsort(PX)
ax.imshow(L[:, order], aspect='auto', cmap='RdBu_r', vmin=-np.abs(L).max()/3, vmax=np.abs(L).max()/3)
ax.set_xlabel('cortex points (sorted left → right)'); ax.set_ylabel('electrodes')
ax.set_title('Lead field matrix L\n(each column = one cortex point)')

for j, k in enumerate(['V1', 'rOFA', 'rFFA']):
    ax = fig.add_subplot(2, 4, 2 + j)
    col = L[:, REGION_IDX[k]]
    mne.viz.plot_topomap(col, info_eeg, axes=ax, show=False, cmap='RdBu_r',
                         vlim=(-np.abs(col).max(), np.abs(col).max()), sphere=SPHERE)
    ax.set_title(f'Column of L for {k}\n(its scalp "fingerprint")')

ax = fig.add_subplot(2, 4, (6, 7))
plot_cortex(ax, np.linalg.norm(L, axis=0), 'Sensitivity: how strongly the electrodes\n"see" each cortex point (norm of its column)',
            cmap='viridis')
ax = fig.add_subplot(2, 4, 8)
corr = np.corrcoef(L[:, [REGION_IDX[k] for k in REGION_IDX]].T)
ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(4)); ax.set_xticklabels(REGION_IDX, rotation=45); ax.set_yticks(range(4)); ax.set_yticklabels(REGION_IDX)
for a in range(4):
    for b in range(4):
        ax.text(b, a, f'{corr[a, b]:.2f}', ha='center', va='center', fontsize=7)
ax.set_title('Similarity of scalp fingerprints')
plt.tight_layout(); plt.show()

**How to read it.**
* Each region has its own scalp fingerprint. rOFA and rFFA are both on the right and their fingerprints are **similar** (high correlation on the right). That similarity is the root of the inverse problem: the scalp has trouble telling neighbours apart.
* The sensitivity map shows that points close to the electrodes are "seen" better. Simple inverse methods are therefore biased toward superficial sources, unless corrected.

---
## Step 7. The inverse problem in miniature
**Paper:** "Motivation and background": "for any pattern of sensor values, there could be infinitely many source distributions that would all fit it perfectly."

The forward step is easy: J → Y. The **inverse** step, Y → J, is the one we need, and it has no unique answer. Here it is with 3 cortex points and 2 electrodes:

$$L = \begin{bmatrix}1&1&0\\0&1&1\end{bmatrix},\qquad Y=\begin{bmatrix}2\\2\end{bmatrix}$$

Three unknowns and two measurements, so infinitely many answers. Every inverse method is just a **rule for choosing one**:
* **Minimum norm (MNM/IID):** choose the answer with the smallest total energy. Formula: $\hat J = L^\top (L L^\top)^{-1} Y$.
* **Sparse (MSP-like):** prefer answers where only a few points are active.
* **fMRI-informed:** prefer answers where fMRI says activity is. With a prior covariance *C* (how much we expect each point to be active): $\hat J = C L^\top (L C L^\top)^{-1} Y$.

In [ ]:
Lt = np.array([[1., 1., 0.], [0., 1., 1.]]); Yt = np.array([2., 2.])
cands = {'only middle':          np.array([0., 2., 0.]),
         'only outer two':       np.array([2., 0., 2.]),
         'all equal':            np.array([1., 1., 1.]),
         'minimum norm (MNM)':   Lt.T @ np.linalg.solve(Lt @ Lt.T, Yt)}
C_fmri = np.diag([0.02, 1.0, 0.02])                    # "fMRI says the middle point is active"
cands['with fMRI prior'] = C_fmri @ Lt.T @ np.linalg.solve(Lt @ C_fmri @ Lt.T, Yt)

fig, axes = plt.subplots(1, len(cands), figsize=(13, 2.6), sharey=True)
for ax, (name, J) in zip(axes, cands.items()):
    ax.bar(['S1', 'S2', 'S3'], J, color=['tab:orange', 'tab:red', 'tab:orange'])
    fit = Lt @ J
    ax.set_title(f'{name}\nsensors = {fit.round(2)}\nenergy = {np.sum(J**2):.2f}', fontsize=8)
axes[0].set_ylabel('source strength')
fig.suptitle('Every one of these explains the data (A=2, B=2) perfectly', y=1.15)
plt.show()

**How to read it.** All five bar charts reproduce the sensors exactly. Minimum norm spreads activity over all three points (lowest energy, but blurry). The fMRI prior pushes the answer toward the middle point. Everything that follows is this same idea at full scale.

---
## Step 8. Preparing the data for inversion
**Paper:** this happens inside SPM's inversion ("data reduction"), so you don't see it in the batch.

Two quiet but important operations:
1. **Spatial reduction.** After average referencing, the 64 channels carry only **63** independent numbers (they always sum to zero). We project both the data and **L** into that 63-dimensional space so the maths stays well-behaved.
2. **Pre-whitening.** We estimate the noise covariance from the **pre-stimulus baseline** (−100 to 0 ms, where there is no picture response yet), then rescale the data so that noise becomes "1 everywhere and uncorrelated". This tells the inversion which channel combinations are noisy and should be trusted less.

**Information added:** a **noise model**. The method can only separate signal from noise if it knows what noise looks like.

In [ ]:
labels_code = epochs.events[:, 2]
is_face = np.isin(labels_code, [CONDS['Famous'], CONDS['Unfamiliar']]).astype(int)
X = epochs.get_data(picks='eeg') * 1e6                    # trials x 64 x time, µV
assert list(epochs.copy().pick('eeg').ch_names) == ch_names
times_ep = epochs.times

# 1. remove the one dimension killed by the average reference
Ured = np.linalg.svd(np.eye(n_ch) - 1. / n_ch)[0][:, :n_ch - 1]      # 64 x 63
Xr = np.einsum('ck,nct->nkt', Ured, X)
Lr = Ured.T @ L

# 2. noise covariance from the baseline of single trials, lightly regularised
base = Xr[:, :, times_ep < 0].transpose(1, 0, 2).reshape(n_ch - 1, -1)
Cn = np.cov(base)
Cn_reg = Cn + 0.1 * np.trace(Cn) / Cn.shape[0] * np.eye(Cn.shape[0])
evals, evecs = np.linalg.eigh(Cn_reg)
Wh = evecs @ np.diag(evals ** -0.5) @ evecs.T                       # whitener
Xw = np.einsum('ij,njt->nit', Wh, Xr)
Lw = Wh @ Lr

# data used to learn the priors: the condition averages after stimulus onset
Y_avg = {c: Xw[labels_code == CONDS[c]].mean(0) for c in CONDS}
post = times_ep >= 0
Y_fit = np.hstack([Y_avg[c][:, post] for c in CONDS])

fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
axes[0].imshow(Cn, cmap='RdBu_r', vmin=-np.abs(Cn).max(), vmax=np.abs(Cn).max()); axes[0].set_title('Noise covariance (before)')
Cw = np.cov(np.einsum('ij,jk->ik', Wh, base))
axes[1].imshow(Cw, cmap='RdBu_r', vmin=-1.5, vmax=1.5); axes[1].set_title('After whitening (≈ identity)')
for a in axes: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()
print('Data for fitting priors:', Y_fit.shape, '(63 whitened channels x time points of 3 condition averages)')

---
## Step 9. The Bayesian inversion engine (Parametric Empirical Bayes)
**Paper:** "In brief, it can be shown that assuming the source activities J are sampled from a multivariate normal distribution… the problem comes down to estimating this covariance matrix… as a weighted sum of covariance components."

This is the heart of SPM's approach. All methods in the paper (MNM, MSP, fMRI priors, group inversion) use **one** engine and differ only in the **list of components** you give it.

**1. The prior on sources** is a covariance matrix *C* built from building blocks $Q_k$, each with a weight $\gamma_k$ (a "hyperparameter"):

$$C = \sum_k \gamma_k Q_k$$

| Method | Components $Q_k$ |
|---|---|
| MNM / IID | one component: identity matrix (every point equally likely, independent) |
| MSP | hundreds of components, each a small cortex **patch** |
| + fMRI priors | one extra component **per fMRI cluster** |

**2. The weights are learned from the data.** Under this model the sensor data have covariance

$$\Sigma = \gamma_{\text{noise}} I + L\,C\,L^\top = \gamma_{\text{noise}} I + \sum_k \gamma_k\, L Q_k L^\top$$

We choose the weights γ that make the observed data **most probable** (maximise the *evidence*). Components that don't help explain the data are pushed to **γ ≈ 0** and effectively switched off. That is why the paper calls fMRI priors "soft": a wrong fMRI cluster gets discounted automatically.

**3. Once γ is known, the source estimate is**

$$\hat J = C L^\top \Sigma^{-1} Y \qquad(\text{the same formula as the toy in Step 7})$$

**4. Model evidence (free energy)** scores each set of assumptions as roughly **accuracy − complexity**. Here we use the log-likelihood minus a penalty for each active component, a simplified stand-in for SPM's free energy.

Below, γ is fitted with a standard **ARD update** (Wipf & Nagarajan, 2009). SPM uses a related ReML / Greedy Search scheme; the idea is identical.

In [ ]:
def make_components(Lw, specs):
    """specs: list of (name, kind, vector). kind = 'noise' | 'iid' | 'diag' | 'rank1'."""
    comps = []
    for name, kind, vec in specs:
        if kind == 'noise':  B = np.eye(Lw.shape[0])                # sensor noise
        elif kind == 'iid':  B = Lw                                  # identity over all sources
        elif kind == 'diag': B = Lw * np.sqrt(vec)[None, :]          # independent sources inside a region (fMRI cluster)
        elif kind == 'rank1': B = Lw @ vec[:, None]                  # one coherent smooth patch (MSP)
        comps.append(dict(name=name, kind=kind, vec=vec, B=B, A=B @ B.T))   # A = L Q L^T in sensor space
    return comps

def ard_fit(Y, comps, n_iter=300, tol=1e-7, keep_history=False):
    """Learn the weights gamma_k that maximise the evidence p(Y | gamma)."""
    n, T = Y.shape
    S = Y @ Y.T / T
    g = np.array([np.trace(S) / (len(comps) * np.trace(c['A'])) for c in comps])
    hist = [g.copy()]
    for it in range(n_iter):
        Sigma = sum(gk * c['A'] for gk, c in zip(g, comps))
        Si = np.linalg.inv(Sigma); SiY = Si @ Y
        g_new = np.array([g[k] * np.sqrt(np.sum((c['B'].T @ SiY) ** 2) / (T * np.sum(Si * c['A'])))
                          for k, c in enumerate(comps)])
        change = np.max(np.abs(np.log(g_new + 1e-300) - np.log(g + 1e-300)))
        g = g_new
        if keep_history: hist.append(g.copy())
        if change < tol: break
    Sigma = sum(gk * c['A'] for gk, c in zip(g, comps))
    logdet = np.linalg.slogdet(Sigma)[1]
    logL = -0.5 * T * (logdet + np.trace(np.linalg.solve(Sigma, S)) + n * np.log(2 * np.pi))
    contrib = np.array([gk * np.trace(c['A']) for gk, c in zip(g, comps)])
    n_active = int(np.sum(contrib > 1e-3 * np.trace(Sigma)))
    F = logL - 0.5 * n_active * np.log(T)            # accuracy - complexity
    return dict(gamma=g, Sigma=Sigma, logL=logL, F=F, n_active=n_active,
                n_iter=it + 1, history=np.array(hist), comps=comps)

def inverse_operator(Lw, fit):
    """K such that J_hat = K @ Y (sources x whitened channels)."""
    C = np.zeros((Lw.shape[1], Lw.shape[1]))
    for gk, c in zip(fit['gamma'], fit['comps']):
        if c['kind'] == 'iid':   C[np.diag_indices_from(C)] += gk
        elif c['kind'] == 'diag': C[np.diag_indices_from(C)] += gk * c['vec']
        elif c['kind'] == 'rank1': C += gk * np.outer(c['vec'], c['vec'])
    return C @ Lw.T @ np.linalg.inv(fit['Sigma'])

def power_maps(K, Y_by_cond, t, window=(0.10, 0.25)):
    """Source power (mean of squared estimate) inside a time window for each condition average."""
    m = (t >= window[0]) & (t <= window[1])
    return {c: np.mean((K @ Y[:, m]) ** 2, axis=1) for c, Y in Y_by_cond.items()}

RESULTS = {}

---
## Step 10. Inversion 1: Minimum norm (MNM / IID)
**Paper:** "The first one corresponds to a L2-minimum norm… called MNM or IID in SPM… conceptually very similar to classical MNE, except that the degree of regularization is estimated."

**Components:** sensor noise + one identity matrix over all cortex points.

**What to expect:** the estimate is **smooth and spread out**. Minimum norm explains the data with the lowest total energy, so it smears activity across many points rather than concentrating it.

In [ ]:
specs_mnm = [('noise', 'noise', None), ('IID', 'iid', None)]
fit = ard_fit(Y_fit, make_components(Lw, specs_mnm))
fit['K'] = inverse_operator(Lw, fit)
RESULTS['MNM'] = fit
print(f"MNM: converged in {fit['n_iter']} iterations, gamma = {dict(zip(['noise','IID'], fit['gamma'].round(6)))}")

def show_inversion(name):
    fit = RESULTS[name]; K = fit['K']
    pw = power_maps(K, {'faces': (Y_avg['Famous'] + Y_avg['Unfamiliar']) / 2,
                        'scr': Y_avg['Scrambled']}, times_ep)
    d = pw['faces'] - pw['scr']
    truth = sum(REGION_W[k] for k in FACE_AREAS)
    fig = plt.figure(figsize=(13, 3.6))
    ax = fig.add_subplot(1, 4, 1); plot_cortex(ax, truth, 'TRUTH\n(face areas)', colorbar=False)
    ax = fig.add_subplot(1, 4, 2); plot_cortex(ax, pw['faces'], f'{name}: power for faces\n100–250 ms', cmap='hot_r')
    ax = fig.add_subplot(1, 4, 3)
    plot_cortex(ax, d, f'{name}: faces − scrambled power', cmap='RdBu_r', vmin=-np.abs(d).max(), vmax=np.abs(d).max())
    ax = fig.add_subplot(1, 4, 4)       # cf. paper Figure 7 (top): estimated time course at the peak
    i = REGION_IDX['rFFA']
    for c, col in zip(CONDS, ['tab:blue', 'tab:red', 'tab:olive']):
        ax.plot(times_ep * 1000, K[i] @ Y_avg[c], color=col, label=c)
    ax.set_title(f'{name}: estimated activity at rFFA'); ax.set_xlabel('ms'); ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()

show_inversion('MNM')

**How to read it.** Blue circles mark the true face areas. MNM finds activity in the right place **on average**, but as large blurry blobs that spill far beyond the true areas. This is the look of the paper's Figure 8A. The time course at rFFA (right) shows a clear face > scrambled response around 150–200 ms: the timing information EEG is good at.

---
## Step 11. Inversion 2: Multiple Sparse Priors (MSP)
**Paper:** "The second type of inversion… unique to SPM… corresponds to a sparse prior on the sources, namely that only a few are active."

**Components:** sensor noise + **300 smooth patches** spread evenly over the cortex (SPM uses 512). Each patch is a little "candidate source". The engine keeps the few patches that explain the data and pushes the rest to γ ≈ 0. The paper uses Greedy Search (GS) to do this; we use ARD.

**What to expect:** **focal** activity in a few places.

In [ ]:
centres = [0]; dmin = np.linalg.norm(rr - rr[0], axis=1)
for _ in range(299):                                 # farthest-point sampling = evenly spread patch centres
    j = int(np.argmax(dmin)); centres.append(j)
    dmin = np.minimum(dmin, np.linalg.norm(rr - rr[j], axis=1))
PATCHES = [patch(rr, c, sigma=0.008) for c in centres]
specs_patches = [(f'patch{i}', 'rank1', p) for i, p in enumerate(PATCHES)]

fit = ard_fit(Y_fit, make_components(Lw, [('noise', 'noise', None)] + specs_patches))
fit['K'] = inverse_operator(Lw, fit)
RESULTS['MSP'] = fit
g_p = fit['gamma'][1:]
kept = np.argsort(g_p)[::-1][:fit['n_active'] - 1]
print(f"MSP: {fit['n_active'] - 1} of {len(PATCHES)} patches kept (the rest shrunk toward 0)")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
plot_cortex(axes[0], None, 'All 300 candidate patch centres (black)')
axes[0].scatter(PX[centres], PY[centres], s=8, c='k')
plot_cortex(axes[1], np.max([g_p[i] / g_p.max() * PATCHES[i] for i in kept], axis=0),
            'Patches the data kept (weight γ)', cmap='hot_r')
plt.tight_layout(); plt.show()
show_inversion('MSP')

**How to read it.** Only a fraction of the 300 candidate patches survive. Some sit around the true face areas; others explain the **background brain activity** (Step 3), which is also present in the data. That background is the same for faces and scrambled faces, so it mostly cancels in the faces − scrambled map. The face difference is **much more focal** than MNM's, like the paper's Figure 8B. Small blue or red spots away from the circles are leftover background noise.

MSP is not magic: it can choose a patch slightly off the true spot, or miss a weak source. Its strength is a cleaner, more interpretable map *when its assumption (few active areas) is true*.

---
## Step 12. Simulated fMRI, turned into priors
**Paper:** "A second constraint is to use prior spatial information, i.e., significant clusters from the group fMRI analysis… each fMRI cluster will become a separate prior."

**Why fMRI?**

| | Where? | When? |
|---|---|---|
| EEG/MEG | poor (centimetres, ambiguous) | excellent (milliseconds) |
| fMRI | excellent (millimetres) | poor (seconds) |

The idea: let fMRI say **where**, and let EEG say **when**.

**What happens here.** We simulate an fMRI *faces > scrambled* t-map: the three true face areas, spatially smoothed, plus noise. We **deliberately add a false cluster** in the left frontal lobe. This is realistic: fMRI integrates over seconds, so it also sees areas that are active **later** (e.g., at 600 ms during the decision). EEG in our 100–250 ms window has nothing from there. A good method must **ignore** this misleading hint.

We then threshold the map and turn each surviving cluster into one prior component: a "diag" component saying "points in this cluster may be active (independently)".

**Information added:** precise *spatial* information from another imaging modality. This is the "asymmetric integration" in the paper: fMRI guides EEG, but is not itself fitted.

In [ ]:
FALSE_IDX = nearest(rr, (-0.5, 0.6, 0.6))
false_w = patch(rr, FALSE_IDX)
D = np.exp(-np.linalg.norm(rr[:, None] - rr[None], axis=2) ** 2 / (2 * 0.004 ** 2))
D /= D.sum(1, keepdims=True)                                # spatial smoothing kernel

bold = D @ (sum(REGION_W[k] for k in FACE_AREAS) + 0.8 * false_w)
noise = D @ rng.standard_normal(N); noise /= noise.std()
fmri_t = 9 * bold / bold.max() + 1.0 * noise
T_THR = 3.5
fmri_clusters = [c for c in clusters(rr, fmri_t > T_THR) if len(c) >= 5]

def cluster_name(c):
    cands = {**{k: REGION_IDX[k] for k in FACE_AREAS}, 'frontal (FALSE)': FALSE_IDX}
    return min(cands, key=lambda k: np.linalg.norm(rr[cands[k]] - rr[c].mean(0)))

FMRI_NAMES = [cluster_name(c) for c in fmri_clusters]
fmri_masks = []
for c in fmri_clusters:
    m = np.zeros(N); m[c] = 1; fmri_masks.append(m)
print('fMRI clusters found:', [(n, len(c)) for n, c in zip(FMRI_NAMES, fmri_clusters)])

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
plot_cortex(axes[0], fmri_t, 'Simulated fMRI t-map\n(faces > scrambled)', cmap='RdBu_r', vmin=-9, vmax=9)
plot_cortex(axes[1], fmri_t, f'Thresholded at t > {T_THR}', cmap='hot_r', mask=fmri_t > T_THR, vmin=T_THR)
lab = np.full(N, np.nan)
for j, c in enumerate(fmri_clusters): lab[c] = j
plot_cortex(axes[2], lab, 'Clusters → one prior component each', cmap='tab10', vmin=-0.5, vmax=9.5, colorbar=False)
for n, c in zip(FMRI_NAMES, fmri_clusters):
    i = c[np.argmax(fmri_t[c])]
    axes[2].text(PX[i], PY[i] + 0.15, n, ha='center', fontsize=8)
# show the frontal cluster too, from above (it is outside the back view)
inset = axes[2].inset_axes([0.72, 0.62, 0.3, 0.38])
qx, qy = flat(rr, centre=(0, 0, 1))
inset.scatter(qx, qy, c='0.88', s=3); inset.scatter(qx, qy, c=lab, cmap='tab10', vmin=-0.5, vmax=9.5, s=4)
inset.set_title('from above', fontsize=7); inset.set_aspect('equal'); inset.axis('off')
plt.tight_layout(); plt.show()

---
## Step 13. Inversions 3 and 4: adding the fMRI priors
**Paper:** "We can combine group-based optimisation of fMRI priors…" (inversion indices 3 and 4).

* **MNM + fMRI:** noise + IID + one component per fMRI cluster.
* **MSP + fMRI:** noise + 300 patches + one component per fMRI cluster.

Watch the weights γ of the fMRI components as the engine learns. The true clusters should stay strong, and the **false frontal cluster should shrink toward zero**. The paper (citing Henson et al., 2010) says exactly this: "invalid fMRI priors were generally discounted, but valid priors were kept."

(The paper also uses *group* inversion, assuming all 16 subjects share the same sources. With one subject we skip that, but it is the same idea: one more constraint that reduces ambiguity.)

In [ ]:
specs_fmri = [(f'fMRI: {n}', 'diag', m) for n, m in zip(FMRI_NAMES, fmri_masks)]

fit = ard_fit(Y_fit, make_components(Lw, specs_mnm + specs_fmri), keep_history=True)
fit['K'] = inverse_operator(Lw, fit); RESULTS['MNM + fMRI'] = fit
fit2 = ard_fit(Y_fit, make_components(Lw, [('noise', 'noise', None)] + specs_patches + specs_fmri))
fit2['K'] = inverse_operator(Lw, fit2); RESULTS['MSP + fMRI'] = fit2

# learning curves of the fMRI weights (MNM + fMRI)
h = fit['history']; names = [c['name'] for c in fit['comps']]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for j, n in enumerate(names):
    if n.startswith('fMRI') or n == 'IID':
        contrib = h[:, j] * np.trace(fit['comps'][j]['A'])
        axes[0].semilogy(contrib, label=n, lw=2 if 'FALSE' in n else 1, ls='--' if 'FALSE' in n else '-')
axes[0].set_xlabel('iteration'); axes[0].set_ylabel('variance explained by component (log)')
axes[0].set_title('MNM + fMRI: learning the prior weights γ'); axes[0].legend(fontsize=7)
final = {n: g * np.trace(c['A']) for n, g, c in zip(names, fit['gamma'], fit['comps']) if n.startswith('fMRI')}
axes[1].barh(list(final), np.log10(np.array(list(final.values())) + 1e-30),
             color=['tab:gray' if 'FALSE' in n else 'tab:red' for n in final])
axes[1].set_xlabel('log10 contribution'); axes[1].set_title('Final weight of each fMRI prior')
plt.tight_layout(); plt.show()

show_inversion('MNM + fMRI')
show_inversion('MSP + fMRI')

**How to read it.**
* **Learning curves:** the false frontal prior (dashed) collapses by many orders of magnitude, while the true clusters keep a strong weight. The data "vetoed" the wrong hint. This is the "soft prior / automatic relevance determination" from the paper.
* **MNM + fMRI** is much sharper than plain MNM: activity concentrates in the fMRI-supported face areas (cf. Figure 9A versus 8A).
* **MSP + fMRI** changes little compared to MSP, because MSP was already focal. The paper found the same: "for the MSP inversion however, the addition of fMRI priors does not help much."

---
## Step 14. Model evidence: which set of assumptions is best?
**Paper:** "One important innovation of SPM's source reconstruction code is the ability to compare different model assumptions (e.g. MNM vs. MSP) in terms of their Bayesian model evidence."

All four inversions fit the **same data**, so we can ask which set of assumptions makes the data most probable, after penalising complexity. Higher is better. A difference of more than ~3 in log-evidence is conventionally "strong evidence".

**Information added:** an **objective** way to choose between methods for *this* dataset, instead of picking by taste.

In [ ]:
names = list(RESULTS)
F = np.array([RESULTS[n]['F'] for n in names])
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(names, F - F.min(), color=['tab:blue', 'tab:orange', 'tab:cyan', 'tab:red'])
ax.set_ylabel('log-evidence relative to worst')
ax.set_title('Model comparison (higher = better)')
plt.xticks(rotation=15); plt.tight_layout(); plt.show()
for n in names:
    print(f"{n:12s}  log-evidence ≈ {RESULTS[n]['F']:10.1f}   active components = {RESULTS[n]['n_active']}")

**How to read it.** In our simulation the truth really is "a few focal areas", so sparse models (MSP) win, and adding true fMRI priors helps minimum norm a lot. The same pattern appears in the paper for subject 15 ("this MSP solution has a higher model evidence").

Our penalty is simpler than SPM's free energy and counts correlated time points as independent, so trust the **ranking** more than the exact numbers.

---
## Step 15. Source power in the time-frequency window and statistics (Figures 8 & 9)
**Paper:** "Time-frequency contrasts" (100–250 ms, 10–20 Hz) and "Group statistics on source reconstructions".

**What happens here.**
1. Band-pass the clean continuous data at **10–20 Hz** and take the **Hilbert envelope**. Squared, this gives instantaneous power. Working per trial captures both *evoked* power (time-locked) and *induced* power (not time-locked).
2. Push every **single trial** through each inverse operator *K* to get power at every cortex point, averaged over **100–250 ms**.
3. Compare **faces vs scrambled** at every cortex point with a t-test.
4. **Correct for multiple comparisons.** We test ~1,200 points, so some would pass p < 0.05 by luck alone. SPM uses Random Field Theory for this. We use the simpler **max-T permutation test**: shuffle the face/scrambled labels 1,000 times, record the largest t each time, and keep only points whose real t beats 95% of those maxima. This controls the family-wise error (FWE) at 0.05, like the paper's "p < 0.05 FWE corrected".

In the paper, step 3 compares **16 subjects**; here, with one subject, it compares **trials**. The logic is the same.

**Information added:** **statistical confidence**, telling apart "where the effect reliably is" from "where it might be noise".

In [ ]:
# 1. 10-20 Hz analytic signal from continuous data (long epochs avoid edge effects, then crop)
raw_band = raw_pre.copy().filter(10., 20., picks=['eeg'], method='iir', iir_params=dict(order=4, ftype='butter'))
ep_band = mne.Epochs(raw_band, events_200, event_id=CONDS, tmin=-0.5, tmax=0.9, baseline=None, preload=True)
ep_band = ep_band[np.isin(ep_band.selection, epochs.selection)]          # same good trials as before
assert len(ep_band) == len(epochs)
Xb = ep_band.get_data(picks='eeg') * 1e6
Hb = signal.hilbert(Xb, axis=2)
win = (ep_band.times >= 0.10) & (ep_band.times <= 0.25)
Hw = np.einsum('ij,jk,nkt->nit', Wh, Ured.T, Hb[:, :, win])            # reduce + whiten, window only

def welch_t(a, b):
    return (a.mean(0) - b.mean(0)) / np.sqrt(a.var(0, ddof=1) / len(a) + b.var(0, ddof=1) / len(b))

def maxT_test(Pw, labels, n_perm=1000, seed=0):
    r = np.random.default_rng(seed)
    t_obs = welch_t(Pw[labels == 1], Pw[labels == 0])
    maxs = np.array([welch_t(Pw[l == 1], Pw[l == 0]).max()
                     for l in (r.permutation(labels) for _ in range(n_perm))])
    p_fwe = (1 + (maxs[None, :] >= t_obs[:, None]).sum(1)) / (n_perm + 1)
    return t_obs, np.quantile(maxs, 0.95), p_fwe

STATS = {}
for name, fit in RESULTS.items():
    Pw = np.mean(np.abs(np.einsum('sk,nkt->nst', fit['K'], Hw)) ** 2, axis=2)    # trials x sources
    STATS[name] = maxT_test(Pw, is_face)
    print(f'{name:12s}: FWE threshold t = {STATS[name][1]:.2f}, significant points = {(STATS[name][0] > STATS[name][1]).sum()}')

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 3.8))
plot_cortex(axes[0], sum(REGION_W[k] for k in FACE_AREAS), 'TRUTH', colorbar=False)
for ax, (name, (t_obs, thr, p)) in zip(axes[1:], STATS.items()):
    sig = t_obs > thr
    plot_cortex(ax, t_obs, f'{name}\nT > {thr:.2f} (p<0.05 FWE)', cmap='autumn_r', mask=sig, vmin=thr)
fig.suptitle('Faces > Scrambled, 10–20 Hz power, 100–250 ms (cf. paper Figures 8 and 9)', y=1.02)
plt.tight_layout(); plt.show()

# SPM-style results table
print(f"{'model':12s} {'cluster':>7s} {'size':>5s} {'peak T':>7s} {'p(FWE)':>7s} {'x,y,z (mm)':>16s}  nearest true area")
for name, (t_obs, thr, p) in STATS.items():
    for j, c in enumerate(clusters(rr, t_obs > thr)[:5]):
        i = c[np.argmax(t_obs[c])]
        dist = {k: np.linalg.norm(rr[i] - rr[REGION_IDX[k]]) * 1000 for k in FACE_AREAS}
        best = min(dist, key=dist.get)
        xyz = ','.join(f'{v:4d}' for v in np.round(rr[i] * 1000).astype(int))
        print(f'{name:12s} {j+1:7d} {len(c):5d} {t_obs[i]:7.2f} {p[i]:7.3f} {xyz:>16s}  {best} ({dist[best]:.0f} mm away)')
    print()

**How to read it** (compare with the paper's Figures 8 and 9 and the table under them):
* **MNM:** large significant regions around the face areas, spreading well beyond them (Figure 8A's "broad right fusiform cluster").
* **MSP:** smaller, more focal clusters closer to OFA/FFA (Figure 8B).
* **MNM + fMRI:** the clusters shrink onto the fMRI-supported face areas (Figure 9A), and **nothing** appears at the false frontal cluster.
* **MSP + fMRI:** similar to MSP. The fMRI prior adds little when the method is already sparse (Figure 9B).
* The **table** mirrors SPM's: cluster size, peak T, corrected p-value and coordinates. With a simulation we can also say how far each peak is from the truth, which you can never do with real data.
* An isolated small spot far from any true area is **cross-talk** (leakage): the inverse operator smears a little of every real source everywhere. With 100+ trials, even a tiny but consistent leak can become "significant". Statistics tell you an effect is *reliable*, not that it is *located exactly there*.

---
## Summary: what each step contributed

| Step | Name | Information it adds | If you skip it… |
|---|---|---|---|
| 1 | Head model & coregistration | Where sensors are relative to brain (from **sMRI**) | Source maps land in the wrong place |
| 4 | Filtering / re-reference | Nothing new; removes drift, mains, reference bias | Noise swamps the response |
| 5 | Epoch, reject, average, contrast | **Timing** of the face effect and its scalp map | No visible N170 |
| 6 | Forward model **L** | **Physics + anatomy**: how each brain point looks at the scalp | No way to go from scalp to brain |
| 7–9 | Inverse problem + PEB engine | A principled rule to pick one solution and learn its settings | Infinitely many answers |
| 10 | MNM | Smooth, cautious estimate | (baseline method) |
| 11 | MSP | Sparse estimate: "few areas are active" | Blurry maps |
| 12–13 | fMRI priors | **Where** from fMRI; wrong hints are switched off automatically | MNM stays blurry |
| 14 | Model evidence | An objective choice between assumptions | Choosing by taste |
| 15 | Power + statistics | **Confidence**, corrected for testing many points | Can't separate effects from noise |

**One sentence:** EEG tells you *when*, MRI tells you *where the sensors and cortex are*, fMRI tells you *where to look*, and the Bayesian inversion combines all three, while letting the EEG data veto any hint that doesn't fit.

### Things to try
* Set the false fMRI cluster to be **very** strong (change `0.8 * false_w` to `3 * false_w`). Is it still rejected?
* Remove a **true** face area from the fMRI (e.g., delete the rFFA cluster from `fmri_masks`). Can MNM + fMRI still find rFFA through the IID component?
* Halve `N_PER` (fewer trials) and watch the statistics weaken.
* Make the sources deeper (shell radius 0.05 instead of 0.07) and watch localisation get harder.
* MNE's own ready-made versions of these ideas: `mne.minimum_norm` (MNE/dSPM/sLORETA) and `mne.inverse_sparse.gamma_map` (a hierarchical Bayesian sparse method close to SPM's ARD).